# Mission 05 — Dual-Path GraphRAG, Hybrid Search & NotebookLM Grounding

**USSE Noosphere / Week 05 / Subsystem 05 (+7.1% target)**  
**Complete four-station notebook:** HNSW · BM25/RRF · NotebookLM audit recorder · Prompt-injection defenses/RAG Triad.

This notebook uses demonstrative, **offline sample data**. It does **not** impersonate Google NotebookLM. The source-backed NotebookLM lab must be completed by the group and all empirical citations/audio must be checked against real sources.


In [ ]:
import importlib.util
import json
from pathlib import Path
import numpy as np

ROOT = Path.cwd()
if not (ROOT / '01_HNSW_Vector_Index.py').exists():
    if (ROOT / 'mission-05' / '01_HNSW_Vector_Index.py').exists():
        ROOT = ROOT / 'mission-05'
    else:
        raise FileNotFoundError('Run notebook inside the mission-05 folder, and retain 01_HNSW_Vector_Index.py.')

def load_file(filename, module_name):
    spec = importlib.util.spec_from_file_location(module_name, ROOT / filename)
    obj = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(obj)
    return obj

hnsw = load_file('01_HNSW_Vector_Index.py', 'mission5_hnsw')
hybrid = load_file('02_BM25_and_RRF_Combiner.py', 'mission5_hybrid')
notebooklm = load_file('03_NotebookLM_RAG_Audit.py', 'mission5_notebooklm')
security = load_file('04_RAG_Provenance_and_Safety.py', 'mission5_security')
print('All four mission modules loaded.')


## Station 1 — Pure NumPy HNSW multilayer proximity index (25 points)

Cosine similarity $\mathrm{sim}(u,v) = u\cdot v/(\|u\|_2\|v\|_2)$; each new node's maximum layer $\ell = \lfloor -\ln(U)/\ln(M)floor$. Multi-layer greedy routing descends to `search_layer(..., ef, layer)` at the bottom. Building and querying are approximate; **$O(\log N)$ is an expected search-structure intuition, not a deterministic timing guarantee**.


In [ ]:
X = np.array([[1,0], [0,1], [0.9,0.1], [-1,0]], dtype=np.float32)
index = hnsw.HNSWIndex(dim=2, M=4, ef_construction=16, ef_search=16, seed=7).add_many(X)
q = np.array([1,0], dtype=np.float32)
print('Query:', q.tolist())
print('HNSW top-3:', index.search(q, k=3))
print('Exact flat top-3:', hnsw.flat_search(X, q, k=3))
assert [x[0] for x in index.search(q, k=3)] == [x[0] for x in hnsw.flat_search(X, q, k=3)]
print('Graph nodes by layer:', [len(g) for g in index.graph])


In [ ]:
# Fast verified benchmark on this notebook host.
smoke_benchmark = hnsw.benchmark(n=250, dim=32, queries=4, k=5, seed=42)
print(json.dumps(smoke_benchmark, indent=2))

# Required full-size experiment; enable deliberately because pure NumPy/Python building 25,000
# 768-D nodes may be very expensive. No fabricated timing numbers are supplied.
RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    full_result = hnsw.benchmark(n=25_000, dim=768, queries=10, k=5)
    (ROOT / 'results').mkdir(exist_ok=True)
    (ROOT / 'results' / 'hnsw_25k_benchmark.json').write_text(json.dumps(full_result, indent=2))
    print(json.dumps(full_result, indent=2))
else:
    print('Full N=25,000 / dim=768 benchmark NOT RUN. Set RUN_FULL_BENCHMARK=True to measure.')


## Station 2 — BM25 + dense HNSW + RRF (25 points)

Okapi BM25 uses $k_1 = 1.5$ and $b = 0.75$. We tokenize exact serial codes such as `XR-4821-A` as one term. RRF uses $\mathrm{RRF}(d) = \sum_{m\in\{dense,sparse\}} 1/(60 + \mathrm{rank}_m(d))$ with 1-based ranks.

**Embedding limitation:** We use deterministic hashed word vectors with a few declared synonym groups, not a pretrained semantic model. Replace `HashingEmbedder` with a trained encoder for genuine conceptual retrieval experiments.


In [ ]:
corpus = hybrid.DEMO_CORPUS
bm25 = hybrid.BM25Index(corpus)
print('Code tokens:', hybrid.tokenize('Serial: XR-4821-A'))
print('BM25 exact match:', bm25.search('XR-4821-A', k=3))
retriever = hybrid.HybridRetriever(corpus)
conceptual = retriever.search('How do navigation and embeddings find conceptually related passages?', k=3)
exact = retriever.search('XR-4821-A', k=3)
print('Conceptual query:', json.dumps(conceptual, indent=2))
print('Exact-code query:', json.dumps(exact, indent=2))
assert exact['hybrid'][0]['id'] == 'K01'
assert hybrid.reciprocal_rank_fusion([['a', 'b'], ['b', 'a']], k=60)[0][1] > 0


## Station 3 — NotebookLM citation/negative/audio audit (25 points)

Upload the original three PDFs (Lewis et al. 2020, Edge et al. 2024, USSE archive) to [NotebookLM](https://notebooklm.google.com). Use the **5 cross-document probes**, **3 negative queries**, click every badge to verify the highlighted source text **actually entails** each claim, and transcribe a real **~2-minute two-speaker Audio Overview**. Citation precision = true entailed citations / all generated citation badges.

The following cell prints the protocol and *unmeasured* scores. Do not use a fictional transcript or pretend source citations were clicked.


In [ ]:
observations_path = ROOT / 'data' / 'notebooklm_observations.json'
observations = json.loads(observations_path.read_text(encoding='utf-8'))
print('Required documents:')
for d in observations['documents']:
    print('-', d['id'], d['title'], 'uploaded=', d['uploaded'])
print('\n5 cross-document probes:')
for p in observations['cross_document']:
    print(p['id'], p['question'])
print('\n3 negative probes:')
for p in observations['negative_constraints']:
    print(p['id'], p['question'])
metrics = notebooklm.evaluate(observations)
print('\nNotebookLM measured status:', json.dumps(metrics, indent=2))
assert metrics['status'] in ('needs_human_observations', 'complete')


## Station 4 — Indirect prompt injection & RAG Triad / provenance (15 points)

Treat retrieved passages as **untrusted**. Detect instruction override text, exfiltration patterns, role delimiters and suspicious embedding matches. Unsafe retrieved chunks are **discarded**. Automated context relevance, faithfulness and answer relevance are **heuristic proxies**, not logical proof or a deployed security guarantee.


In [ ]:
record = security.audit_turn(
    'What is the coolant pump part identifier?',
    security.DEMO_CHUNKS,
    log_path=ROOT / 'results' / 'provenance.jsonl',
)
print(json.dumps({k: record[k] for k in (
    'query', 'retrieved_chunk_ids', 'accepted_chunk_ids', 'blocked_chunk_ids',
    'final_grounded_answer', 'rag_triad')}, indent=2))
assert 'attack-02' in record['blocked_chunk_ids']
assert 'attack-02' not in record['accepted_chunk_ids']
assert all(0 <= record['rag_triad'][metric] <= 1 for metric in (
    'context_relevance', 'faithfulness', 'answer_relevance'))


## Integration checkpoint and the four station sign-offs (10 points)

- [ ] **Systems Architect:** verify HNSW implementation + execute **25,000 × 768** full benchmark; sign with name, date and Git commit.
- [ ] **Protocol Officer:** verify BM25 exact codes and RRF(k=60); sign with name/date/commit.
- [ ] **Data Specialist:** upload 3 originals, answer 5 grounded and 3 negative probes, click every citation, inspect real audio and fill data; sign with name/date/commit.
- [ ] **Security Sentinel:** review injection cases, Triad limitations and provenance JSONL; sign with name/date/commit.

**Submission status:** source code implemented; NotebookLM audit and full-size benchmark still require authentic user-run validation. This notebook makes no invented empirical claims. See `notebooklm_audit_report.md` and `README.md`.
